# 07 Fact Checker Agent

In [1]:
# ==================================================
# Notebook 07
# Fact Checker Agent
# ==================================================

import re
import numpy as np
import pandas as pd

from typing import TypedDict
from typing import List
from typing import Dict

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
class ResearchState(TypedDict):

    research_goal: str

    queries_executed: List[str]

    raw_findings: List[str]

    deduplicated_findings: List[str]

    missing_topics: List[str]

    conflicts: List[Dict]

    critic_score: float

    trust_score: float

    draft_report: str

    final_report: str

    iteration_count: int

    status: str

    errors: List[str]

    metadata: Dict

In [4]:
findings = [
    "AWS cloud spending increased 12%",
    "AWS cloud spending decreased 8%",
    "Azure cloud spending remained stable",
    "Google Cloud storage costs dropped 4%",
    "Google Cloud storage costs decreased 4%",
]

In [5]:
POSITIVE_TERMS = ["increase", "increased", "growth", "grew", "rise", "rose", "up"]

NEGATIVE_TERMS = [
    "decrease",
    "decreased",
    "decline",
    "declined",
    "drop",
    "dropped",
    "down",
]

In [6]:
def detect_direction(text):

    text = text.lower()

    for word in POSITIVE_TERMS:

        if word in text:
            return "positive"

    for word in NEGATIVE_TERMS:

        if word in text:
            return "negative"

    return "neutral"

In [7]:
detect_direction("AWS spending increased 12%")

'positive'

In [8]:
def extract_entity(text):

    entities = ["AWS", "Azure", "Google Cloud", "Oracle Cloud", "IBM Cloud"]

    for entity in entities:

        if entity.lower() in text.lower():

            return entity

    return None

In [9]:
extract_entity("AWS cloud spending increased 12%")

'AWS'

In [10]:
def detect_conflict(statement_a, statement_b):

    entity_a = extract_entity(statement_a)

    entity_b = extract_entity(statement_b)

    if entity_a != entity_b:

        return False

    direction_a = detect_direction(statement_a)

    direction_b = detect_direction(statement_b)

    if direction_a == "positive" and direction_b == "negative":

        return True

    if direction_a == "negative" and direction_b == "positive":

        return True

    return False

In [11]:
detect_conflict("AWS cloud spending increased 12%", "AWS cloud spending decreased 8%")

True

In [12]:
def semantic_similarity(text_a, text_b):

    emb_a = embedding_model.encode(text_a)

    emb_b = embedding_model.encode(text_b)

    score = cosine_similarity(emb_a.reshape(1, -1), emb_b.reshape(1, -1))[0][0]

    return float(score)

In [13]:
def discover_conflicts(findings, threshold=0.60):

    conflicts = []

    for i in range(len(findings)):

        for j in range(i + 1, len(findings)):

            sim = semantic_similarity(findings[i], findings[j])

            if sim < threshold:

                continue

            if detect_conflict(findings[i], findings[j]):

                conflicts.append(
                    {
                        "statement_a": findings[i],
                        "statement_b": findings[j],
                        "similarity": round(sim, 3),
                    }
                )

    return conflicts

In [14]:
conflicts = discover_conflicts(findings)

pd.DataFrame(conflicts)

,statement_a,statement_b,similarity
0,AWS cloud spending increased 12%,AWS cloud spending decreased 8%,0.901


In [15]:
def compute_trust_score(findings, conflicts):

    if len(findings) == 0:

        return 0.0

    score = 1 - (len(conflicts) / len(findings))

    return round(max(score, 0), 2)

In [16]:
compute_trust_score(findings, conflicts)

0.8

In [17]:
def fact_checker_agent(state: ResearchState):

    print("\nFact Checker Agent Running...")

    findings = state["deduplicated_findings"]

    conflicts = discover_conflicts(findings)

    trust_score = compute_trust_score(findings, conflicts)

    state["conflicts"] = conflicts

    state["trust_score"] = trust_score

    state["status"] = "fact_checked"

    return state

In [18]:
sample_state = {
    "research_goal": "Cloud Cost Analysis",
    "queries_executed": [],
    "raw_findings": [],
    "deduplicated_findings": findings,
    "missing_topics": [],
    "conflicts": [],
    "critic_score": 0.0,
    "trust_score": 0.0,
    "draft_report": "",
    "final_report": "",
    "iteration_count": 0,
    "status": "",
    "errors": [],
    "metadata": {},
}

In [19]:
updated_state = fact_checker_agent(sample_state)


Fact Checker Agent Running...


In [20]:
pd.DataFrame(updated_state["conflicts"])

,statement_a,statement_b,similarity
0,AWS cloud spending increased 12%,AWS cloud spending decreased 8%,0.901


In [21]:
updated_state["trust_score"]

0.8

In [22]:
def conflict_severity(state):

    count = len(state["conflicts"])

    if count == 0:

        return "Low"

    elif count <= 2:

        return "Medium"

    return "High"

In [23]:
conflict_severity(updated_state)

'Medium'

In [24]:
from langgraph.graph import StateGraph, END

In [25]:
graph = StateGraph(ResearchState)

graph.add_node("fact_checker", fact_checker_agent)

graph.set_entry_point("fact_checker")

graph.add_edge("fact_checker", END)

app = graph.compile()

In [26]:
result = app.invoke(sample_state)


Fact Checker Agent Running...
